# El Niño / food-prices — English audio render (Don's voice, F5) + post-process

Renders the **English** narration of `blog/drafts/el-nino-food-prices-2026/` in your cloned voice
(F5-TTS), runs the full **post-process chain** (`scripts/audio-post-process.mjs` — warmth EQ /
de-ess / light compress / **−16 LUFS EBU R128** / pink-noise bed / optional branded intro), then
**commits + pushes** the finished `audio.mp3` + `audio.json`.

English only, by design (Spanish text ships for parity; Spanish audio is a later pass). If Colab
disconnects mid-run you lose at most the article in flight — re-run cells **1–6** then the render
cell; it skips anything already pushed.

First: `Runtime → Change runtime type → T4 GPU → Save`, then run top to bottom (Shift+Enter).
Your voice reference already lives in the repo — nothing to upload.

## 1. Confirm GPU
Output should mention `Tesla T4` (or similar).

In [ ]:
!nvidia-smi

## 2. Install Node 20 + F5-TTS + ffmpeg (~2 min)
`ffmpeg` drives **both** the render (WAV concat + MP3) and the post-process; F5-TTS clones the voice.

In [ ]:
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
!apt-get install -y nodejs ffmpeg -qq > /dev/null
!node --version && ffmpeg -version | head -1
!pip install -q f5-tts jieba

## 3. Add your GitHub PAT — use the 🔑 **Secrets** panel
Left sidebar → **key (🔑) icon → "+ Add new secret"**. Name it exactly `GITHUB_PAT`, paste a token
with **repo** scope, and turn **Notebook access** ON. The token is read from there, never written
into the notebook. (No Secrets panel? A hidden paste-prompt appears below the cell when you run it.)

In [ ]:
import os, subprocess
BRANCH = 'claude/strategic-council-board-docs-m3w6dy'
REPO   = 'github.com/Donwonmagic/potentially-profitable.git'

token = None
try:
    from google.colab import userdata
    token = (userdata.get('GITHUB_PAT') or '').strip()
    if token: print('using token from the Secrets panel (GITHUB_PAT)')
except Exception:
    token = None
if not token:
    import getpass
    token = getpass.getpass('Paste GitHub token (repo scope) here, then Enter: ').strip()
assert token, 'no token — add GITHUB_PAT to the Secrets panel (step 3) and re-run'

# Re-run safe: step to /content before wiping the clone (the kernel CWD may still
# be inside the old clone from a previous run).
os.chdir('/content')
url = f'https://x-access-token:{token}@{REPO}'
!rm -rf /content/potentially-profitable
subprocess.run(['git','clone','-b',BRANCH,'--depth','1',url,'/content/potentially-profitable'], check=True)
%cd /content/potentially-profitable
!git config user.name 'Don Goldstein'
!git config user.email 'dongoldstein.accts@gmail.com'
# F5 hardcodes torch.xpu (Intel GPU); Colab's CUDA build lacks it. Make the check safe.
import glob
for f in glob.glob('/usr/local/lib/python3*/dist-packages/f5_tts/**/*.py', recursive=True):
    s = open(f, encoding='utf-8').read()
    if 'torch.xpu.is_available()' in s:
        open(f,'w',encoding='utf-8').write(s.replace('torch.xpu.is_available()', '(hasattr(torch,"xpu") and torch.xpu.is_available())'))
print('cloned', BRANCH, '+ F5 patched')

## 4. Verify your voice reference is in the repo

In [ ]:
import os
assert os.path.isfile('scripts/voice-refs/don-reference.m4a'), 'voice clip missing'
assert os.path.isfile('scripts/voice-refs/don-reference.txt'), 'transcript missing'
print('voice reference present:', os.path.getsize('scripts/voice-refs/don-reference.m4a'), 'bytes')

## 5. Pick the target(s)
Defaults to the El Niño draft. It lives under `blog/drafts/`, so the general library scan wouldn't
catch it — that's why it's listed explicitly. Add more paths (published `blog/<slug>` or
`library/<slug>`) to render a batch in the same run.

In [ ]:
TARGETS = [
    'blog/drafts/el-nino-food-prices-2026',
    # 'blog/some-other-post',
]
import os
for d in TARGETS:
    idx = os.path.join(d, 'index.html')
    assert os.path.isfile(idx), f'missing: {idx}'
    assert 'id="listen-btn"' in open(idx, encoding='utf-8').read(), f'no listen button in {idx}'
    print('ready:', d)

## 6. Render → post-process → commit, per article
Per target: render the English track in your voice, run the post-process chain on `audio.mp3`,
then commit + push **just** `audio.mp3` + `audio.json` (never the `.raw.mp3` backup). A failure on
one article cleans up its half-written output and moves on. The F5 base model (~1.5 GB) downloads
on the first article.

**Resume after a disconnect:** re-run cells 1–6, then this cell (already-pushed articles are skipped).

In [ ]:
import os, json, subprocess, time

def is_done(d):
    j = os.path.join(d, 'audio.json')
    if not (os.path.isfile(j) and os.path.isfile(os.path.join(d, 'audio.mp3'))): return False
    try: m = json.load(open(j, encoding='utf-8'))
    except Exception: return False
    return str(m.get('engine','')) == 'f5'

def run(cmd): return subprocess.run(cmd).returncode

def commit_push(d):
    run(['git','add', f'{d}/audio.mp3', f'{d}/audio.json'])
    if subprocess.run(['git','diff','--cached','--quiet']).returncode == 0: return True
    if run(['git','commit','-m', f'audio: {d} (EN)']) != 0: return False
    for a in range(4):
        if run(['git','push','origin',BRANCH]) == 0: return True
        time.sleep(2**(a+1))
    return False

def cleanup(d):
    for f in ('audio.mp3','audio.json','audio.raw.mp3'):
        p = os.path.join(d, f)
        if os.path.isfile(p): os.remove(p)

done = skipped = failed = 0; fails = []
for i, d in enumerate(TARGETS, 1):
    if is_done(d):
        skipped += 1; print(f'[{i}/{len(TARGETS)}] skip (already f5): {d}'); continue
    print(f'[{i}/{len(TARGETS)}] render: {d}', flush=True)
    if run(['node','scripts/render-post-audio.mjs', d, '--engine','f5','--languages','en','--force-retranslate']) != 0:
        failed += 1; fails.append(d); cleanup(d); print('  !! render failed — cleaned up, continuing'); continue
    print(f'    post-process: {d}/audio.mp3', flush=True)
    if run(['node','scripts/audio-post-process.mjs', f'{d}/audio.mp3']) != 0:
        failed += 1; fails.append(d); cleanup(d); print('  !! post-process failed — cleaned up, continuing'); continue
    if not commit_push(d):
        failed += 1; fails.append(d); cleanup(d); print('  !! commit/push failed — cleaned up, continuing'); continue
    done += 1; print(f'    committed + pushed: {d}')
print(f'\n=== rendered {done}, skipped {skipped}, failed {failed} ===')
if fails: print('failed (retry on next run):', fails)

## 7. Done
Each target above is committed + pushed to the working branch as `audio: <dir> (EN)` — the
post-processed MP3 + its `audio.json` manifest. The live player picks up the new audio on the next
deploy; when the post is published (`git mv blog/drafts/... blog/...`), `audio.mp3` moves with it.

**If the cloned voice sounds over-produced**, re-run the post-process cell with `--no-bed` (drop the
ambient bed) and/or `--no-intro` (skip the branded opener) appended to the `audio-post-process.mjs`
command. **Spanish audio, later:** the same loop with `--languages es` plus the F5-Spanish
checkpoint (see `scripts/voice-refs/README.md`).